# Demo 04a — Maximum Likelihood and MAP

**Class 04 · Block 1** (slides: `slides/04_probabilistic_models.md`)

Five live demos: where the losses of Class 03 come from, and what a prior adds.

* **M1 — MLE for a coin (Bernoulli):** the likelihood, the log-likelihood, and `jax.grad` ascent.
* **M2 — MLE for a Gaussian:** the mean and the variance, and why the variance is biased.
* **M3 — MLE *is* the loss:** Gaussian noise gives MSE, Bernoulli labels give cross-entropy, Laplace noise gives MAE.
* **M4 — MAP for a coin (Beta prior):** the prior fades with data; Laplace smoothing is a Beta prior.
* **M5 — MAP for linear regression (Gaussian prior):** ridge regression, and a posterior with uncertainty.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import log_loss
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

## M1 — Maximum likelihood for a coin

A coin lands heads with an unknown probability $\theta$. We toss it $n = 10$ times and see $k = 7$ heads. The
**likelihood** of $\theta$ is the probability of the data *as a function of the parameter*:

$$L(\theta) = \theta^{k}(1-\theta)^{n-k}, \qquad \ell(\theta) = \log L(\theta) = k\log\theta + (n-k)\log(1-\theta)$$

The **maximum likelihood estimate** is the parameter that makes the data most probable. Setting
$\ell'(\theta) = \frac{k}{\theta} - \frac{n-k}{1-\theta} = 0$ gives $\hat\theta = k/n$: **counting**.

In [ ]:
n, k = 10, 7
theta = np.linspace(0.001, 0.999, 500)
lik = theta**k * (1 - theta) ** (n - k)
loglik = k * np.log(theta) + (n - k) * np.log(1 - theta)
print(f"MLE from the curve: {theta[loglik.argmax()]:.3f}  (k / n = {k / n})   max likelihood {lik.max():.5f}")

fig, axes = plt.subplots(1, 2)
axes[0].plot(theta, lik)
axes[0].axvline(k / n, color="tab:red", ls="--")
axes[0].set(title="likelihood L(theta)", xlabel=r"$\theta$")
axes[1].plot(theta, loglik)
axes[1].axvline(k / n, color="tab:red", ls="--")
axes[1].set(title="log-likelihood", xlabel=r"$\theta$", ylim=(-25, 0))
plt.show()

**With JAX.** We do not need the formula: write the negative log-likelihood and let `jax.grad` do the derivative. To keep
$\theta \in (0,1)$, optimize the **logit** $\phi$, with $\theta = \sigma(\phi)$.

In [ ]:
def neg_loglik(phi, k, n):
    p = jax.nn.sigmoid(phi)
    return -(k * jnp.log(p) + (n - k) * jnp.log1p(-p))


grad = jax.jit(jax.grad(neg_loglik))
phi = 0.0
for step in range(200):
    phi = phi - 0.1 * grad(phi, k, n)
print(f"gradient descent on the negative log-likelihood: theta = {float(jax.nn.sigmoid(phi)):.4f}")

**Small data misleads.** Three tosses, three heads: the MLE is $\hat\theta = 1$: "tails is impossible". MLE has no
way to say "I have seen too little" (the same problem as a word never seen in spam, next block).

In [ ]:
for n_, k_ in ((3, 3), (10, 7), (100, 70), (1000, 700)):
    print(f"n = {n_:4d}, k = {k_:3d}: MLE = {k_ / n_:.3f}")

## M2 — Maximum likelihood for a Gaussian

Data $x_1,\dots,x_n \sim \mathcal{N}(\mu, \sigma^2)$. The negative log-likelihood is

$$-\ell(\mu,\sigma) = \frac{n}{2}\log(2\pi\sigma^2) + \frac{1}{2\sigma^2}\sum_i (x_i - \mu)^2
\quad\Longrightarrow\quad \hat\mu = \bar x, \quad \hat\sigma^2 = \frac1n\sum_i (x_i - \bar x)^2$$

The MLE variance divides by $n$, not $n-1$.

In [ ]:
x = jnp.array([2.1, 1.9, 2.6, 2.4, 3.0])


def gauss_nll(params, x):
    mu, log_sigma = params
    sigma2 = jnp.exp(2 * log_sigma)
    return 0.5 * x.size * jnp.log(2 * jnp.pi * sigma2) + jnp.sum((x - mu) ** 2) / (2 * sigma2)


g = jax.jit(jax.grad(gauss_nll))
params = jnp.array([0.0, 0.0])
for _ in range(3000):
    params = params - 0.02 * g(params, x)
mu_hat, var_hat = float(params[0]), float(jnp.exp(2 * params[1]))
print(f"gradient descent: mu = {mu_hat:.4f}, variance = {var_hat:.4f}")
print(f"closed form     : mean = {float(x.mean()):.4f}, variance (divide by n) = {float(x.var()):.4f}, (divide by n-1) = {float(x.var(ddof=1)):.4f}")

In [ ]:
# Is the MLE variance biased? Average it over many samples of size n from N(0, 1).
rng = np.random.default_rng(0)
for n_ in (3, 5, 10, 50):
    samples = rng.normal(0, 1, (100_000, n_))
    print(f"n = {n_:2d}: mean of the MLE variance {samples.var(axis=1).mean():.3f}   (theory (n-1)/n = {(n_ - 1) / n_:.3f})   unbiased {samples.var(axis=1, ddof=1).mean():.3f}")

The MLE **underestimates** the variance by the factor $(n-1)/n$: it measures the spread around the *sample* mean, which
is closer to the data than the true mean. The bias vanishes with $n$. Gaussian Naive Bayes (next demo) uses the MLE
variance, and adds a small floor for safety.

## M3 — The loss is a negative log-likelihood

| Model of the noise | Loss | Class |
|:--|:--|:--|
| $y \sim \mathcal{N}(\hat y, \sigma^2)$ | squared error (MSE) | linear regression |
| $y \sim \text{Laplace}(\hat y, b)$ | absolute error (MAE) | robust regression |
| $y \sim \text{Bernoulli}(\sigma(z))$ | cross-entropy | logistic regression |
| $y \sim \text{Categorical}(\text{softmax}(z))$ | softmax cross-entropy | multi-class |

**Gaussian noise $\Rightarrow$ MSE.** Fit a line by minimizing the Gaussian negative log-likelihood with the noise
level $\sigma$ also a parameter: the weights equal least squares, and $\hat\sigma^2$ is the MSE.

In [ ]:
housing = pl.read_csv(DATA / "california_housing.csv.zst")
for c in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
    housing = housing.with_columns(pl.col(c).log())
names_h = [c for c in housing.columns if c != "MedHouseVal"]
X_h = housing.select(names_h).to_numpy().astype(float)
y_h = housing["MedHouseVal"].to_numpy().astype(float)
perm = np.random.default_rng(1).permutation(len(y_h))
te_h, tr_h = perm[: len(y_h) // 5], perm[len(y_h) // 5 :]
scaler_h = StandardScaler().fit(X_h[tr_h])
Xtr_h, Xte_h = np.asarray(scaler_h.transform(X_h[tr_h])), np.asarray(scaler_h.transform(X_h[te_h]))
ytr_h, yte_h = y_h[tr_h], y_h[te_h]


def nll_gauss_linear(params, X, y):
    w, b, log_sigma = params["w"], params["b"], params["log_sigma"]
    sigma2 = jnp.exp(2 * log_sigma)
    resid = y - (X @ w + b)
    return 0.5 * jnp.log(2 * jnp.pi * sigma2) + jnp.mean(resid**2) / (2 * sigma2)     # mean over examples


vg = jax.jit(jax.value_and_grad(nll_gauss_linear))
params = {"w": jnp.zeros(8), "b": jnp.zeros(()), "log_sigma": jnp.zeros(())}
m = jax.tree.map(jnp.zeros_like, params)
v = jax.tree.map(jnp.zeros_like, params)
for t in range(1, 4001):                                                          # Adam
    _, gr = vg(params, Xtr_h, ytr_h)
    m = jax.tree.map(lambda a, g_: 0.9 * a + 0.1 * g_, m, gr)
    v = jax.tree.map(lambda a, g_: 0.999 * a + 0.001 * g_**2, v, gr)
    step = 0.05 * 0.5 * (1 + np.cos(np.pi * (t - 1) / 4000))
    params = jax.tree.map(lambda p, a, b_: p - step * (a / (1 - 0.9**t)) / (jnp.sqrt(b_ / (1 - 0.999**t)) + 1e-8), params, m, v)

A = np.c_[Xtr_h, np.ones(len(Xtr_h))]
lstsq = np.linalg.lstsq(A, ytr_h, rcond=None)[0]
mse_train = float(np.mean((ytr_h - A @ lstsq) ** 2))
print("max |weights: MLE - least squares| =", float(np.abs(np.r_[np.asarray(params["w"]), float(params["b"])] - lstsq).max()))
print(f"MLE noise variance {float(jnp.exp(2 * params['log_sigma'])):.4f}  vs  training MSE {mse_train:.4f}")

**Bernoulli labels $\Rightarrow$ cross-entropy.** The mean negative log-likelihood of a probabilistic classifier *is*
scikit-learn's `log_loss`.

In [ ]:
rng = np.random.default_rng(0)
p = rng.uniform(0.05, 0.95, 1000)
y = (rng.uniform(size=1000) < p).astype(int)
nll = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
print(f"mean negative log-likelihood {nll:.6f}   sklearn log_loss {log_loss(y, p):.6f}")

**Laplace noise $\Rightarrow$ MAE (robust).** With heavy-tailed noise the likelihood punishes outliers linearly, not
quadratically. On a line with two gross outliers:

In [ ]:
rng = np.random.default_rng(3)
xs = np.linspace(0, 10, 30)
ys = 1.0 + 0.5 * xs + rng.normal(0, 0.3, 30)
ys[[5, 25]] += 8                                                       # two gross outliers


def fit(loss):
    def total(params):
        return loss(ys - (params[0] * xs + params[1]))
    g_ = jax.jit(jax.grad(total))
    p_ = jnp.array([0.0, 0.0])
    for _ in range(5000):
        p_ = p_ - 0.02 * g_(p_)
    return np.asarray(p_)


w_mse = fit(lambda r: jnp.mean(r**2))
w_mae = fit(lambda r: jnp.mean(jnp.sqrt(r**2 + 1e-6)))                # a smooth |r|
print("true line: slope 0.50, intercept 1.00")
print(f"Gaussian noise (MSE): slope {w_mse[0]:.3f}, intercept {w_mse[1]:.3f}")
print(f"Laplace noise  (MAE): slope {w_mae[0]:.3f}, intercept {w_mae[1]:.3f}")
plt.scatter(xs, ys, s=12, color="k")
plt.plot(xs, w_mse[0] * xs + w_mse[1], label="MSE fit (Gaussian)")
plt.plot(xs, w_mae[0] * xs + w_mae[1], label="MAE fit (Laplace)")
plt.legend()
plt.show()

## M4 — MAP: a prior on the parameter

Bayes' rule for the parameter: $p(\theta \mid D) \propto p(D \mid \theta)\,p(\theta)$. For the coin, the natural
prior is a **Beta** distribution, $p(\theta) \propto \theta^{a-1}(1-\theta)^{b-1}$. It is *conjugate*: the posterior is
again a Beta,

$$\theta \mid D \sim \text{Beta}(a + k,\; b + n - k)$$

$$\hat\theta_{MAP} = \frac{k + a - 1}{n + a + b - 2} \qquad\qquad
\hat\theta_{mean} = \frac{k + a}{n + a + b}$$

$a - 1$ and $b - 1$ act as **imaginary tosses** seen before the data. With $a = b = 2$: one imaginary head and one
tail: $\hat\theta_{MAP} = (k+1)/(n+2)$. This is **Laplace smoothing**.

In [ ]:
rows = []
for n_, k_ in ((3, 3), (10, 7), (30, 21), (300, 210)):
    row = {"n": n_, "k": k_, "MLE": round(k_ / n_, 3)}
    for a, b in ((1, 1), (2, 2), (10, 10)):
        row[f"MAP Beta({a},{b})"] = round((k_ + a - 1) / (n_ + a + b - 2), 3)
    row["post. mean Beta(2,2)"] = round((k_ + 2) / (n_ + 4), 3)
    rows.append(row)
print(pl.DataFrame(rows))

In [ ]:
theta = np.linspace(0.001, 0.999, 500)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (n_, k_) in zip(axes, ((0, 0), (3, 3), (30, 21))):
    for (a, b), color in (((1, 1), "tab:gray"), ((2, 2), "tab:blue"), ((10, 10), "tab:red")):
        ax.plot(theta, stats.beta.pdf(theta, a + k_, b + n_ - k_), color=color, label=f"prior Beta({a},{b})")
    ax.axvline(0.7, color="k", ls=":")
    ax.set(title=f"posterior after n = {n_}, k = {k_}" if n_ else "priors (before any data)", xlabel=r"$\theta$")
axes[0].legend(fontsize=7)
plt.show()

The prior matters for $n = 3$ and fades by $n = 300$: a *strong* prior (Beta(10,10)) needs more data to be overcome.

**Laplace smoothing is a MAP estimate.** For one word that appeared in $k$ of $n$ spam e-mails, `BernoulliNB(alpha=1)`
estimates $\theta = (k+1)/(n+2)$: the Beta(2,2) mode. In general `alpha` $= a - 1$.

In [ ]:
n_, k_ = 8, 0                                                          # a word never seen in 8 spam e-mails
Xw = np.array([[1]] * k_ + [[0]] * (n_ - k_) + [[1]] * 8 + [[0]] * 2)  # + 10 ham e-mails, the word in 8 of them
yw = np.array([1] * n_ + [0] * 10)
for alpha in (1e-10, 1.0, 3.0):
    nb = BernoulliNB(alpha=alpha).fit(Xw, yw)
    print(f"alpha = {alpha:<6}: P(word | spam) = {np.exp(nb.feature_log_prob_[1, 0]):.4f}   MAP formula (k + alpha)/(n + 2 alpha) = {(k_ + alpha) / (n_ + 2 * alpha):.4f}")

## M5 — MAP for regression: ridge is a Gaussian prior

Linear regression with Gaussian noise $\mathcal{N}(0, \sigma^2)$ and a Gaussian prior $w_j \sim \mathcal{N}(0, \tau^2)$:

$$-\log p(w \mid D) = \frac{1}{2\sigma^2}\sum_i (y_i - w^\top x_i)^2 + \frac{1}{2\tau^2}\lVert w\rVert^2 + \text{const}
\;\;\Longrightarrow\;\; \text{ridge with } \alpha = \frac{\sigma^2}{\tau^2}$$

(in scikit-learn's sum convention, $\alpha = \lambda N$). Take the housing data with only $N = 100$ examples.

In [ ]:
sub = np.random.default_rng(5).choice(len(ytr_h), 100, replace=False)
Xs, ys = Xtr_h[sub], ytr_h[sub]
ys_c = ys - ys.mean()                                                  # centre: no intercept to worry about
sigma2, tau2 = 0.4, 0.25
alpha = sigma2 / tau2


def neg_log_posterior(w):
    return jnp.sum((ys_c - Xs @ w) ** 2) / (2 * sigma2) + jnp.sum(w**2) / (2 * tau2)


g = jax.jit(jax.grad(neg_log_posterior))
w = jnp.zeros(8)
for _ in range(10_000):
    w = w - 0.001 * g(w)
ridge = Ridge(alpha=alpha, fit_intercept=False).fit(Xs, ys_c)
print("max |MAP by jax.grad - Ridge(alpha = sigma^2 / tau^2)| =", float(np.abs(np.asarray(w) - ridge.coef_).max()))

**A full posterior, not a point.** For this model the posterior of $w$ is Gaussian:
$\Sigma = \big(X^\top X/\sigma^2 + I/\tau^2\big)^{-1}$, mean $\mu = \Sigma X^\top y/\sigma^2$ (the ridge solution). It
gives an **uncertainty** for every prediction: $\text{var}(\hat y_*) = \sigma^2 + x_*^\top \Sigma\, x_*$.

In [ ]:
Sigma = np.linalg.inv(Xs.T @ Xs / sigma2 + np.eye(8) / tau2)
mu = Sigma @ Xs.T @ ys_c / sigma2
print("posterior mean equals the ridge solution:", float(np.abs(mu - ridge.coef_).max()) < 1e-10)
print(pl.DataFrame({"feature": names_h, "posterior mean": mu.round(3), "posterior std": np.sqrt(np.diag(Sigma)).round(3)}))

pred = Xte_h @ mu + ys.mean()
sd = np.sqrt(sigma2 + np.einsum("ij,jk,ik->i", Xte_h, Sigma, Xte_h))
print(f"test RMSE {np.sqrt(np.mean((yte_h - pred) ** 2)):.3f}; a 90% interval (mean +- 1.645 sd) covers {np.mean(np.abs(yte_h - pred) < 1.645 * sd):.3f} of the test districts")

## Summary

* **MLE** picks the parameter that makes the data most probable; for Bernoulli and categorical variables it is
  **counting**, for a Gaussian the mean and the (biased) variance.
* The losses of Class 03 are negative log-likelihoods: Gaussian noise $\to$ MSE, Laplace $\to$ MAE, Bernoulli $\to$
  cross-entropy.
* **MAP** adds a prior: Beta(2,2) on a probability gives **Laplace smoothing**; a Gaussian prior on weights gives
  **ridge**; a Laplace prior gives lasso.
* With a full posterior, predictions come with uncertainty; the prior fades as data grows.